# Le choix de la fenêtre d'observation

**Objectif :** choisir une fenêtre d’observation conciliant la disponibilité des informations et la précocité de la prédiction de l’abandon.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src" / "config.py").is_file():
    raise RuntimeError("Ouvrir ce notebook depuis la racine du projet ou notebooks/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from src.config import CLEAN_ACTIVITES, CLEAN_EVALUATIONS, CLEAN_INSCRIPTIONS
from src.cleaning import load_tables, merge_tables
from src.preprocessing import j30_eligibility

## 1. Préparation des données

### 1.1 Chargement des données

In [ ]:
activities_df = pd.read_parquet(CLEAN_ACTIVITES)
evaluations_df = pd.read_parquet(CLEAN_EVALUATIONS)
inscriptions_df = pd.read_parquet(CLEAN_INSCRIPTIONS)

### 1.2 Jointure des tables d'événements

Les jours sont mesurés depuis le début du **programme** ($T_0$), et non depuis la date d’inscription.

In [ ]:
cols_ins = ['id_inscription', 'date_debut', 'date_debut_programme', 'date_annulation', 'semestre', 'niveau_scolaire']
activities_merged = activities_df.merge(inscriptions_df[cols_ins], on='id_inscription', how='left')
evaluations_merged = evaluations_df.merge(inscriptions_df[cols_ins], on='id_inscription', how='left')

In [ ]:
activities_merged['days_since_start'] = (activities_merged['horodatage'] - activities_merged['date_debut_programme']).dt.days
evaluations_merged['days_since_start'] = (evaluations_merged['date_soumission'] - evaluations_merged['date_debut_programme']).dt.days

print("Jours écoulés depuis le début du programme — activités :")
print(activities_merged['days_since_start'].describe())
print("\nJours écoulés depuis le début du programme — évaluations :")
print(evaluations_merged['days_since_start'].describe())

## 2. Visualisation

In [ ]:
sns.histplot(data=activities_merged, x='days_since_start', hue='semestre', kde=True)
plt.xlabel('Jours depuis le début du programme (T₀)')
plt.ylabel('Fréquence des activités')
plt.show()

In [ ]:
sns.histplot(data=evaluations_merged, x='days_since_start', hue='semestre', kde=True)
plt.xlabel('Jours depuis le début du programme (T₀)')
plt.ylabel('Fréquence des évaluations')
plt.show()

In [ ]:
print("Activités pré-rentrée (days_since_start < 0) :", (activities_merged['days_since_start'] < 0).sum())
print("Évaluations pré-rentrée (days_since_start < 0) :", (evaluations_merged['days_since_start'] < 0).sum())

In [ ]:
print(activities_merged.groupby('semestre', observed=False)['days_since_start'].describe())
print(evaluations_merged.groupby('semestre', observed=False)['days_since_start'].describe())

## 3. Temps avant abandon

### 3.1 Distribution du délai avant abandon

In [ ]:
student_churn = inscriptions_df[inscriptions_df['resultat_final'] == 'Abandon'].copy()
student_churn['lifespan'] = (student_churn['date_annulation'] - student_churn['date_debut_programme']).dt.days
print(student_churn.groupby('semestre', observed=False)['lifespan'].describe())
print("\nQuantiles globaux de lifespan (jours depuis T_0) :")
print(student_churn['lifespan'].quantile([0.05, 0.10, 0.15, 0.20, 0.25, 0.50, 0.75]))

### 3.2 Visualisation

In [ ]:
sns.histplot(data=student_churn, x='lifespan', hue='semestre', kde=True)
plt.xlabel('Jours écoulés depuis le début du programme (T₀)')
plt.ylabel('Fréquence')
plt.title('Délai avant abandon selon le semestre')
plt.show()

## 4. Comparaison des fenêtres d’observation

### 4.1 Vérification de la troncature liée à l'horizon des données

Une fenêtre trop longue peut dépasser la période couverte par les données. La dernière activité observée fournit ici un repère descriptif, sans prouver à elle seule un suivi complet pour chaque inscription.

In [ ]:
data_cutoff = activities_df['horodatage'].max()
print("Date max d'activité :", data_cutoff)

In [ ]:
students_df = inscriptions_df.copy()
for T_obs in [15, 30, 45, 60, 90]:
    students_df['window_end'] = students_df['date_debut_programme'] + pd.Timedelta(days=T_obs)
    truncated = (students_df['window_end'] > data_cutoff).sum()
    print(f"Fenêtre de {T_obs} jours : {truncated} inscriptions au-delà de la dernière activité observée")

### 4.2 Couverture des événements et abandons déjà enregistrés

Pour chaque fenêtre (15, 30, 45, 60 et 90 jours), on mesure la part des inscriptions non annulées sans activité ou sans évaluation, ainsi que la part des abandons déjà enregistrés. Ce diagnostic inclut les interactions pré-rentrée et porte sur les inscriptions non annulées, avant le filtre d’éligibilité du modèle J30.

In [ ]:
results = []
for T_obs in [15, 30, 45, 60, 90]:
    active_ins = inscriptions_df[
        inscriptions_df['date_annulation'].isna()
        | ((inscriptions_df['date_annulation'] - inscriptions_df['date_debut_programme']).dt.days > T_obs)
    ]
    n_active = len(active_ins)

    activities_window = activities_merged[
        (activities_merged['days_since_start'] <= T_obs)
        & (activities_merged['id_inscription'].isin(active_ins['id_inscription']))
    ]
    activity_summary = activities_window.groupby('id_inscription').agg(
        total_activities=('id_activite', 'count'),
    )

    evaluations_window = evaluations_merged[
        (evaluations_merged['days_since_start'] <= T_obs)
        & (evaluations_merged['id_inscription'].isin(active_ins['id_inscription']))
    ]
    evaluation_summary = evaluations_window.groupby('id_inscription').agg(
        total_evaluations=('id_evaluation', 'count')
    )

    n_activity = activity_summary.shape[0]
    n_evaluation = evaluation_summary.shape[0]

    results.append({
        'T_obs': T_obs,
        'active_enrollments': n_active,
        'no activity % (active)': round(100 * (1 - n_activity / n_active), 2),
        'no evaluation % (active)': round(100 * (1 - n_evaluation / n_active), 2),
        'median activity': activity_summary['total_activities'].median(),
        'median evaluation': evaluation_summary['total_evaluations'].median(),
        'already churned %': round(100 * (student_churn['lifespan'] <= T_obs).mean(), 2),
    })

diagnostics_df = pd.DataFrame(results)
display(diagnostics_df)

fig, ax = plt.subplots()
for column, label in [
    ("no activity % (active)", "Sans activité (inscriptions non annulées)"),
    ("no evaluation % (active)", "Sans évaluation (inscriptions non annulées)"),
    ("already churned %", "Abandons déjà enregistrés (parmi les abandons)"),
]:
    ax.plot(diagnostics_df["T_obs"], diagnostics_df[column], marker="o", label=label)
ax.axvline(30, color="grey", linestyle=":")
ax.set(xlabel="Jour d’observation depuis le début du programme", ylabel="Proportion (%)",
       xticks=diagnostics_df["T_obs"], ylim=(0, 100))
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
evaluations_merged['deadline_from_start'] = (
    evaluations_merged['date_echeance'] - evaluations_merged['date_debut_programme']
).dt.days
first_eval_deadline = evaluations_merged.groupby('id_inscription')['deadline_from_start'].min()
print("Première échéance d'évaluation (jours depuis T_0) :")
print(first_eval_deadline.describe())

## 5. Écart entre la dernière interaction et l’annulation

$\Delta_i = T_i^{\text{cancel}} - T_i^{\text{last\_interaction}}$

In [ ]:
raw = load_tables()
raw_ins = raw["inscriptions"]
raw_act = raw["activites_virtuelles"]
raw_eval = raw["evaluations"]
enrollment_history = merge_tables(raw_ins, raw["etudiants"], raw["programmes"])

valid_eval = raw_eval.merge(raw_ins[['id_inscription', 'date_debut']], on='id_inscription', how='left')
valid_eval = valid_eval[
    ~(
        (valid_eval['date_soumission'] < valid_eval['date_debut'])
        | (valid_eval['date_echeance'] < valid_eval['date_debut'])
    )
]

last_act_date = raw_act.groupby('id_inscription')['horodatage'].max().rename('last_act_date')
last_eval_date = valid_eval.groupby('id_inscription')['date_soumission'].max().rename('last_eval_date')

enrollment_history = enrollment_history.merge(last_act_date, on='id_inscription', how='left')
enrollment_history = enrollment_history.merge(last_eval_date, on='id_inscription', how='left')
enrollment_history['last_interaction_date'] = enrollment_history[['last_act_date', 'last_eval_date']].max(axis=1)

# Cohorte exploratoire : conserver les annulations manquantes pour en mesurer la fréquence.
pre_t0_cancel = enrollment_history['date_annulation'].notna() & (
    enrollment_history['date_annulation'] < enrollment_history['date_debut_programme']
)
label_mismatch = enrollment_history['date_annulation'].notna() & (enrollment_history['resultat_final'] != 'Abandon')
cohort_post_t0 = enrollment_history[~(pre_t0_cancel | label_mismatch)].copy()

cohort_post_t0['cancel_day'] = (
    cohort_post_t0['date_annulation'] - cohort_post_t0['date_debut_programme']
).dt.days
cohort_post_t0['last_interaction_day'] = (
    cohort_post_t0['last_interaction_date'] - cohort_post_t0['date_debut_programme']
).dt.days

abandons = cohort_post_t0[cohort_post_t0['resultat_final'] == 'Abandon'].copy()
abandons['has_cancel_date'] = abandons['date_annulation'].notna()

print("=== 1. Abandons post-T_0 : Date d'annulation observée vs manquante ===")
print(abandons['has_cancel_date'].value_counts())
print("\nPar semestre :")
print(pd.crosstab(abandons['semestre'], abandons['has_cancel_date'], margins=True))

In [ ]:
confirmed_abandons = abandons[abandons['has_cancel_date']].copy()
confirmed_abandons['delta_days'] = (
    confirmed_abandons['cancel_day'] - confirmed_abandons['last_interaction_day']
)

print("=== 2. Distribution de Delta_i = T_cancel - T_last_interaction (en jours) ===")
print(confirmed_abandons['delta_days'].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))
print(f"\nCas où T_last_interaction > T_cancel (delta_days < 0) : {(confirmed_abandons['delta_days'] < 0).sum()} ({(confirmed_abandons['delta_days'] < 0).mean():.2%})")
print(f"Cas où |delta_days| <= 7 jours : {(confirmed_abandons['delta_days'].abs() <= 7).mean():.2%}")
print(f"Cas où delta_days > 30 jours : {(confirmed_abandons['delta_days'] > 30).mean():.2%}")

print("\n=== 3. Dernier jour d'interaction (depuis T_0) par resultat_final ===")
print(
    cohort_post_t0.groupby('resultat_final', observed=False)['last_interaction_day'].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

## 6. Durée observée et risque d’abandon par palier

La dernière interaction ne correspond pas nécessairement à la fin du programme. La table décrit les abandons enregistrés sur des intervalles de 30 jours, sans estimation formelle de survie tenant compte de la censure.

In [ ]:
persisted = cohort_post_t0[cohort_post_t0['resultat_final'] != 'Abandon'].copy()

prog_horizon = (
    persisted.groupby(['semestre', 'niveau_scolaire'], observed=True)['last_interaction_day']
    .agg(
        n_persisted='count',
        q10=lambda s: np.percentile(s, 10),
        q25=lambda s: np.percentile(s, 25),
        median='median',
        q75=lambda s: np.percentile(s, 75),
        q90=lambda s: np.percentile(s, 90),
        q95=lambda s: np.percentile(s, 95),
        max_day='max',
    )
    .round(1)
)
print("=== 1. Horizon du dernier jour d'interaction des étudiants persistants (Reussite/Echec) ===")
display(prog_horizon)

eval_prog = valid_eval.merge(
    cohort_post_t0[['id_inscription', 'semestre', 'niveau_scolaire', 'date_debut_programme']],
    on='id_inscription',
    how='inner',
)
eval_prog['due_day'] = (eval_prog['date_echeance'] - eval_prog['date_debut_programme']).dt.days
eval_prog['sub_day'] = (eval_prog['date_soumission'] - eval_prog['date_debut_programme']).dt.days

deadline_horizon = (
    eval_prog.groupby(['semestre', 'niveau_scolaire'], observed=True)
    .agg(
        n_evals=('id_evaluation', 'count'),
        max_due_day=('due_day', 'max'),
        q95_sub_day=('sub_day', lambda s: np.percentile(s.dropna(), 95)),
        max_sub_day=('sub_day', 'max'),
    )
    .round(1)
)
print("\n=== 2. Horizon des évaluations (échéances et soumissions en jours depuis T_0) ===")
display(deadline_horizon)

In [ ]:
known_cohort = cohort_post_t0[
    (cohort_post_t0['resultat_final'] != 'Abandon') | (cohort_post_t0['date_annulation'].notna())
].copy()

checkpoints = [30, 60, 90, 120, 150, 180, 210, 240]
hazard_rows = []

for t_k in checkpoints:
    for sem in ['Semestre 1', 'Semestre 2']:
        sub = known_cohort[known_cohort['semestre'] == sem]
        at_risk = sub[sub['cancel_day'].isna() | (sub['cancel_day'] > t_k)]
        events_next_30d = at_risk[
            (at_risk['resultat_final'] == 'Abandon') & (at_risk['cancel_day'] <= t_k + 30)
        ]
        persisted_at_risk = at_risk[at_risk['resultat_final'] != 'Abandon']
        active_after_tk = (persisted_at_risk['last_interaction_day'] > t_k).mean()

        hazard_rows.append({
            'semestre': sem,
            'checkpoint_tk': f"J+{t_k}",
            'window_predicted': f"({t_k}, {t_k+30}]",
            'n_at_risk': len(at_risk),
            'n_churn_next_30d': len(events_next_30d),
            'hazard_rate_%': round(100 * len(events_next_30d) / max(len(at_risk), 1), 2),
            'persisted_active_after_tk_%': round(100 * active_after_tk, 2),
        })

hazard_df = pd.DataFrame(hazard_rows)
print("=== 3. Table de survie et taux de risque (Hazard Rate) par palier de 30 jours ===")
display(hazard_df)

## 7. Éligibilité à J30

Une inscription doit avoir commencé au plus tard à J30 et ne pas être annulée à cette date. Les dates de début manquantes sont exclues. Les motifs d’exclusion peuvent se recouvrir.

In [ ]:
cutoff_j30 = inscriptions_df.date_debut_programme + pd.Timedelta(days=30)
eligible_j30 = j30_eligibility(inscriptions_df)
eligibility_summary = pd.Series({
    "inscriptions_nettoyées": len(inscriptions_df),
    "date_début_manquante": int(inscriptions_df.date_debut.isna().sum()),
    "début_après_J30": int(inscriptions_df.date_debut.gt(cutoff_j30).sum()),
    "annulation_au_plus_tard_J30": int(inscriptions_df.date_annulation.le(cutoff_j30).sum()),
    "éligibles_J30": int(eligible_j30.sum()),
    "abandons_ultérieurs": int(inscriptions_df.loc[eligible_j30, "churn"].sum()),
})
assert inscriptions_df.loc[eligible_j30, "date_debut"].notna().all()
assert not inscriptions_df.loc[eligible_j30, "date_debut"].gt(cutoff_j30[eligible_j30]).any()
assert not inscriptions_df.loc[eligible_j30, "date_annulation"].le(cutoff_j30[eligible_j30]).any()
eligibility_summary

## Conclusion

Les distributions des interactions et du délai avant abandon montrent que les premières semaines sont déterminantes pour la construction des variables. Des activités et des soumissions existent également avant le début du programme.

L’exploration initiale, présentée dans le rapport, indiquait une forte amélioration de la couverture des évaluations entre J15 et J30 (part sans évaluation passant de 83,98 % à 23,24 %). La première échéance d’évaluation avait une médiane de 19 jours. À J60 ou J90, davantage d’abandons ont déjà eu lieu, ce qui réduit le temps disponible pour intervenir. Les diagnostics ci-dessus sont recalculés sur les tables nettoyées (1 600 abandons datés, contre 1 610 lors de l’exploration initiale).

**J30** constitue donc un compromis entre l’information disponible et la précocité de l’alerte. La population éligible au modèle comprend **6 626 inscriptions**, dont **1 204 abandons ultérieurs**. Ce choix n’est pas un optimum prédictif démontré.